# MAI 600 — Module 3: Attention & Architecture Walkthrough
**Student:** Alexandre Contaldi Pasquini | **Course:** MAI 600 Natural Language Processing

**Domain:** AI-Driven CRM & Automation (Axelis AI)

This notebook walks through how a Transformer model uses self-attention to process a short text sample from an AI CRM automation workflow. The goal is to identify token relationships, explain attention behaviors, and connect those observations to how Transformer architecture works internally.

## 1. Text Sample

In [ ]:
# Fictional, non-sensitive text — Axelis AI CRM automation scenario
text = """The AI voice agent completed the outbound call because it detected that the lead had previously submitted a contact form. The system flagged the contact as high priority after scoring it above the qualification threshold. Sara, the voice agent, confirmed the appointment with the prospect and logged the outcome in the CRM. When the webhook failed to trigger, the automation team reviewed the pipeline logs and discovered that the stage name had been changed, which broke the trigger condition. They corrected the configuration and reprocessed the affected leads."""

print(text)
print(f"\nWord count: {len(text.split())}")


### Why this text works for attention analysis

| Item | Detail |
|---|---|
| Domain | AI CRM and automation workflow (Axelis AI) |
| Text type | Fictional operational summary — non-sensitive |
| Key relationships | Pronoun resolution ('it', 'they'), entity tracking (Sara, the webhook, the automation team), cause/effect (stage rename → trigger failure) |
| Why attention matters | The model must resolve what 'it' refers to in two different sentences, track which agent performed which action, and connect the webhook failure to its root cause across sentence boundaries |
| Sensitive data | None — all names and events are fictional |

## 2. Tokenization

In [ ]:
import re

tokens = re.findall(r"\w+|[^\w\s]", text)
print(f"Token count: {len(tokens)}\n")
for i, tok in enumerate(tokens):
    print(f"{i:03d}: {tok}")


In [ ]:
# Identify tokens that may tokenize differently in production LLMs
special_tokens = [
    ("Sara",          "Proper name — may split into 'Sar' + 'a' in some tokenizers"),
    ("webhook",       "Technical compound — may split into 'web' + 'hook'"),
    ("high-priority", "Hyphenated — tokenizer usually splits at the hyphen"),
    ("CRM",           "Acronym — usually one token but behavior varies by model"),
    ("reprocessed",   "Prefixed verb — may split into 're' + 'processed'"),
    ("it",            "Pronoun — same token in two different contexts (lead, contact)"),
    ("they",          "Pronoun — ambiguous: user or automation team depending on sentence"),
]

try:
    import pandas as pd
    import warnings; warnings.filterwarnings("ignore")
    df = pd.DataFrame(special_tokens, columns=["Token / Phrase", "Tokenization Note"])
    display(df)
except:
    for row in special_tokens:
        print(row)


## 3. Important Tokens and Context Clues

In [ ]:
important_tokens = [
    {"Token / Phrase": "AI voice agent",       "Why It Matters": "Main entity performing the outbound call action"},
    {"Token / Phrase": "it (sentence 1)",      "Why It Matters": "Pronoun — should refer to 'lead', not 'agent' or 'call'"},
    {"Token / Phrase": "contact form",         "Why It Matters": "Prior event that triggered the high-priority flag"},
    {"Token / Phrase": "it (sentence 2)",      "Why It Matters": "Pronoun — should refer to 'contact', not 'system'"},
    {"Token / Phrase": "Sara",                 "Why It Matters": "Named entity — coreference with 'voice agent' established earlier"},
    {"Token / Phrase": "the prospect",         "Why It Matters": "Entity receiving the appointment — same person as 'lead'"},
    {"Token / Phrase": "webhook",              "Why It Matters": "Technical system component — subject of the failure chain"},
    {"Token / Phrase": "they",                 "Why It Matters": "Pronoun — refers to 'automation team', not 'Sara' or 'lead'"},
    {"Token / Phrase": "stage name",           "Why It Matters": "Root cause of the trigger failure — long-range dependency from webhook failure"},
    {"Token / Phrase": "affected leads",       "Why It Matters": "Same entity as 'lead' in sentence 1 — tracking across full paragraph"},
]

try:
    import pandas as pd
    display(pd.DataFrame(important_tokens))
except:
    for row in important_tokens:
        print(row)


## 4. Three Attention Behaviors

In [ ]:
attention_behaviors = [
    {
        "Attention Behavior": "Pronoun resolution",
        "Token / Phrase 1":   "it",
        "Token / Phrase 2":   "lead / contact",
        "Why the Relationship Matters": (
            "The word 'it' appears twice in the paragraph with different referents. "
            "In sentence 1, 'it' refers to 'lead' (the agent detected the lead had submitted a form). "
            "In sentence 2, 'it' refers to 'contact' (the system flagged the contact after scoring it). "
            "Self-attention must resolve each instance using surrounding context — the token immediately "
            "before 'it' is different in each case, which changes the resolution."
        )
    },
    {
        "Attention Behavior": "Entity tracking across sentences",
        "Token / Phrase 1":   "Sara / voice agent",
        "Token / Phrase 2":   "confirmed / logged",
        "Why the Relationship Matters": (
            "Sara is introduced as the voice agent in sentence 3. The model must link 'Sara' to "
            "'AI voice agent' from sentence 1 to understand that these refer to the same entity. "
            "Then the verbs 'confirmed' and 'logged' must both be attributed to Sara. "
            "Multi-head attention supports this: one head may track entity identity while another "
            "tracks verb-subject relationships."
        )
    },
    {
        "Attention Behavior": "Cause and effect across sentence boundary",
        "Token / Phrase 1":   "stage name had been changed",
        "Token / Phrase 2":   "webhook failed / broke the trigger",
        "Why the Relationship Matters": (
            "The webhook failure in sentence 4 is caused by the stage rename — a fact that appears "
            "later in the same sentence. The model must connect 'webhook failed' (the effect) to "
            "'stage name had been changed' (the cause) and then to 'broke the trigger condition' "
            "(the mechanism). This is a long-range dependency within a single complex sentence, "
            "which is exactly the type of relationship that self-attention handles better than "
            "recurrent models."
        )
    },
    {
        "Attention Behavior": "Coreference across full paragraph",
        "Token / Phrase 1":   "lead (sentence 1)",
        "Token / Phrase 2":   "affected leads (sentence 5)",
        "Why the Relationship Matters": (
            "The entity 'lead' appears in sentence 1 and is then referred to indirectly as 'the prospect' "
            "in sentence 3, and finally as 'affected leads' in sentence 5. "
            "To understand that these all refer to the same category of entity, the model must maintain "
            "a coherent representation of this entity across the full paragraph — a long-range dependency "
            "that tests the depth of the attention mechanism."
        )
    },
]

try:
    import pandas as pd
    df = pd.DataFrame(attention_behaviors)
    display(df)
except:
    for row in attention_behaviors:
        print(row)


## 5. Q / K / V Explained for This Text

In [ ]:
qkv_table = [
    {
        "Transformer Term": "Query (Q)",
        "What It Means":    "What the current token is looking for",
        "Example from text": "The token 'it' (sentence 1) is looking for its referent — what noun does it replace?"
    },
    {
        "Transformer Term": "Key (K)",
        "What It Means":    "What each token offers as a match signal",
        "Example from text": "'lead' offers a strong key match for 'it' because it is the most recent noun before the pronoun."
    },
    {
        "Transformer Term": "Value (V)",
        "What It Means":    "The information carried forward if the token is relevant",
        "Example from text": "If 'it' attends to 'lead', the semantic properties of 'lead' (a person, a contact, someone who submitted a form) are passed forward to help interpret the rest of the sentence."
    },
    {
        "Transformer Term": "Attention weight",
        "What It Means":    "A score showing how strongly one token uses another",
        "Example from text": "'it' should give a high weight to 'lead' and a lower weight to 'agent' or 'call', because the grammar and context make 'lead' the correct referent."
    },
]

try:
    import pandas as pd
    display(pd.DataFrame(qkv_table))
except:
    for row in qkv_table:
        print(row)


## 6. Manual Attention Relationship Matrix & Heatmap

In [ ]:
labels = [
    "AI agent", "lead", "it (s1)", "contact form",
    "Sara", "prospect", "webhook", "automation team",
    "they", "stage name", "affected leads"
]

# 0=no relationship, 1=weak, 2=moderate, 3=strong
scores = [
    # agent  lead  it1  form  Sara  prosp  hook  team  they  stage  aff
    [  0,    2,    0,   0,    3,    0,     0,    0,    0,    0,     1  ],  # AI agent
    [  2,    0,    3,   2,    0,    2,     0,    0,    0,    0,     3  ],  # lead
    [  0,    3,    0,   2,    0,    0,     0,    0,    0,    0,     0  ],  # it (s1)
    [  0,    2,    2,   0,    0,    0,     0,    0,    0,    0,     0  ],  # contact form
    [  3,    0,    0,   0,    0,    3,     0,    0,    0,    0,     0  ],  # Sara
    [  0,    2,    0,   0,    3,    0,     0,    0,    0,    0,     2  ],  # prospect
    [  0,    0,    0,   0,    0,    0,     0,    3,    2,    3,     0  ],  # webhook
    [  0,    0,    0,   0,    0,    0,     3,    0,    3,    2,     2  ],  # automation team
    [  0,    0,    0,   0,    0,    0,     2,    3,    0,    2,     1  ],  # they
    [  0,    0,    0,   0,    0,    0,     3,    2,    2,    0,     0  ],  # stage name
    [  1,    3,    0,   0,    0,    2,     0,    2,    1,    0,     0  ],  # affected leads
]

try:
    import pandas as pd
    matrix = pd.DataFrame(scores, index=labels, columns=labels)
    display(matrix)
except:
    print(scores)


In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import numpy as np

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(scores, cmap="YlOrRd", vmin=0, vmax=3)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)

    for i in range(len(labels)):
        for j in range(len(labels)):
            val = scores[i][j]
            color = "white" if val >= 2 else "black"
            ax.text(j, i, str(val), ha="center", va="center", fontsize=8, color=color)

    ax.set_title(
        "Manual Attention Relationship Matrix — Axelis AI CRM Paragraph\n"
        "(0=none, 1=weak, 2=moderate, 3=strong)",
        fontweight="bold", pad=12
    )
    plt.colorbar(im, ax=ax, label="Relationship Strength")
    plt.tight_layout()
    plt.savefig("attention_heatmap.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved: attention_heatmap.png")
except Exception as e:
    print("Visualization skipped:", e)


## 7. Transformer Diagram Checklist

In [ ]:
checklist = [
    ("1. Raw text input",                      "✅ — the Axelis AI CRM paragraph"),
    ("2. Tokenization",                        "✅ — split into words and subword units"),
    ("3. Token IDs",                           "✅ — each token mapped to a vocabulary index"),
    ("4. Embeddings",                          "✅ — token IDs become dense vectors"),
    ("5. Positional encoding",                 "✅ — position added to each embedding to preserve order"),
    ("6. Self-attention",                      "✅ — each token compares itself with all others"),
    ("7. Queries, Keys, Values",               "✅ — Q/K/V computed for each token"),
    ("8. Attention weights",                   "✅ — softmax(QKᵀ/√d_k) gives attention scores"),
    ("9. Multi-head attention",                "✅ — multiple heads track different relationships simultaneously"),
    ("10. Feed-forward network",               "✅ — transforms each token representation after attention"),
    ("11. Residual connections + LayerNorm",   "✅ — keeps original information, stabilizes training"),
    ("12. Output probabilities",               "✅ — softmax over vocabulary gives next-token distribution"),
    ("13. Generated / predicted output",       "✅ — most probable token selected and decoded to text"),
]

try:
    import pandas as pd
    df = pd.DataFrame(checklist, columns=["Transformer Component", "Status"])
    display(df)
except:
    for row in checklist:
        print(row)


## 8. Reflection

**1. Which token relationship was easiest to identify?**

The cause-and-effect relationship between 'stage name had been changed' and 'webhook failed' was the most straightforward. The sentence explicitly signals causality with 'which broke the trigger condition' — the model has a strong lexical cue to attend from the effect (webhook failure) back to the cause (stage rename).

**2. Which relationship was most ambiguous?**

The pronoun 'they' in sentence 5. At that point in the paragraph, both 'Sara' and 'automation team' are active entities. A model would need to use sentence position and semantic role to determine that 'they' refers to the automation team — Sara's role was completing the appointment, not fixing pipeline configurations. Without sufficient context or fine-tuning on this domain, a model might misattribute the action.

**3. How does self-attention help the model use context?**

Self-attention allows every token to 'look at' every other token in the sequence simultaneously, rather than processing left-to-right one word at a time. This means the model can directly connect 'it' in sentence 1 to 'lead' several tokens earlier without passing information through intermediate states. For long-range dependencies like the coreference between 'lead' (sentence 1) and 'affected leads' (sentence 5), this is especially important — a recurrent model would have to carry information about 'lead' through every intermediate token before reaching 'affected leads,' with increasing risk of that information fading.

**4. Why is attention useful but not a perfect explanation of model behavior?**

Attention weights show which tokens the model is 'looking at,' but high attention does not always mean the attended token is the one causing the output. A model might assign high attention to a token for gradient or positional reasons unrelated to meaning. Additionally, in multi-head attention, different heads track different aspects of the text simultaneously — the overall model behavior emerges from the combination of all heads and all layers, not from any single attention score.

**5. How would this help design prompts or debug LLM behavior?**

Understanding that models rely on attention to resolve pronouns and track entities suggests that prompts should introduce entities clearly and avoid ambiguous pronouns. In the Axelis AI context, if the system prompt says 'Sara will call the lead and she will log the outcome,' a model with weak coreference resolution might not correctly attribute 'she' to Sara if other entities appear nearby. Making entity references explicit — 'Sara will call the lead. Sara will then log the outcome.' — reduces the model's resolution burden and produces more reliable outputs.

## 9. AI Tool Usage Disclosure

| Item | Detail |
|---|---|
| AI tools used | Claude (claude.ai) |
| How I used them | Helped generate the fictional Axelis AI text sample and suggested the token relationship matrix structure |
| What I verified myself | Reviewed all token relationships, assigned all matrix scores, and wrote the reflection and Q/K/V explanations in my own words |
| What I changed | Expanded the pronoun resolution analysis to cover both instances of 'it' after noticing the pronoun appears twice with different referents — that was an independent observation |
| Confirmation | AI was used as a learning and support tool, not as a replacement for my own work |